# 🧠 07. Grayscale Average (AVG) Ekstraksi Fitur CNN (Layer `svm_features`)

Notebook ini menggunakan model CNN yang telah dilatih pada `06_Grayscale_AVG_Training.ipynb` sebagai **feature extractor**.
Fitur representasi tingkat tinggi berdimensi 512 diekstrak dari layer `svm_features` (GlobalAveragePooling2D) dan disimpan ke disk dalam format `.npz`.


In [1]:
import os
import sys

# Setup CUDA DLL path sebelum import TensorFlow agar cuDNN terdeteksi sempurna
_cuda_bin = r'C:\Users\daffa\anaconda3\envs\CNNgpu\Library\bin'
if os.path.exists(_cuda_bin):
    if _cuda_bin not in os.environ.get('PATH', ''):
        os.environ['PATH'] = _cuda_bin + ';' + os.environ.get('PATH', '')
    if hasattr(os, 'add_dll_directory'):
        try:
            os.add_dll_directory(_cuda_bin)
        except Exception:
            pass

import numpy as np
import tensorflow as tf

# Aktifkan GPU memory growth agar hemat VRAM & mencegah error 'DNN library is not found'
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
except Exception:
    pass



ARTIFACT_DIR = 'cifar10_artifacts/grayscale_avg'
cnn_model_path = os.path.join(ARTIFACT_DIR, 'cnn_model.keras')
print(f"Memuat model CNN AVG terlatih dari: {cnn_model_path}")
cnn_model = tf.keras.models.load_model(cnn_model_path)

from tensorflow.keras.models import Model

def load_cifar10_data(load_train=True, load_test=True):
    (X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
    if load_train:
        X_train = X_train.astype('float32') / 255.0
        y_train = y_train.flatten()
    else:
        X_train, y_train = None, None
    if load_test:
        X_test = X_test.astype('float32') / 255.0
        y_test = y_test.flatten()
    else:
        X_test, y_test = None, None
    return X_train, y_train, X_test, y_test

def transform_grayscale_avg(X):
    return np.mean(X, axis=-1, keepdims=True).astype('float32')

def extract_cnn_features(model, X_data, batch_size=128):
    feature_layer = model.get_layer('svm_features')
    extractor = tf.keras.models.Model(inputs=model.input, outputs=feature_layer.output)
    print(f"Feature Extractor Layer : {feature_layer.name} | Output Shape : {feature_layer.output_shape}")
    total_samples = len(X_data)
    features = []
    for i in range(0, total_samples, batch_size):
        end = min(i + batch_size, total_samples)
        batch = X_data[i:end]
        feats = extractor.predict(batch, batch_size=batch_size, verbose=0)
        features.append(feats)
    features_arr = np.vstack(features)
    print(f"[EKSTRAKSI SELESAI] Extracted Features Shape: {features_arr.shape}")
    return features_arr


Memuat model CNN AVG terlatih dari: cifar10_artifacts/grayscale_avg\cnn_model.keras


## 📥 1. Memuat Data & Mengekstrak Vektor Fitur CNN


In [2]:
X_train_raw, y_train, X_test_raw, y_test = load_cifar10_data()

# Transformasi ke Grayscale AVG
X_train = transform_grayscale_avg(X_train_raw)
X_test  = transform_grayscale_avg(X_test_raw)

print("Mengekstrak fitur CNN dari data training (50.000 sampel)...")
X_train_features = extract_cnn_features(cnn_model, X_train, batch_size=128)

print()
print("Mengekstrak fitur CNN dari data testing (10.000 sampel)...")
X_test_features = extract_cnn_features(cnn_model, X_test, batch_size=128)

print()
print(f"[SELESAI] X_train_features shape: {X_train_features.shape}")
print(f"[SELESAI] X_test_features shape : {X_test_features.shape}")


Mengekstrak fitur CNN dari data training (50.000 sampel)...
Feature Extractor Layer : svm_features | Output Shape : (None, 512)
[EKSTRAKSI SELESAI] Extracted Features Shape: (50000, 512)

Mengekstrak fitur CNN dari data testing (10.000 sampel)...
Feature Extractor Layer : svm_features | Output Shape : (None, 512)
[EKSTRAKSI SELESAI] Extracted Features Shape: (10000, 512)

[SELESAI] X_train_features shape: (50000, 512)
[SELESAI] X_test_features shape : (10000, 512)


## 💾 2. Menyimpan Vektor Fitur ke Disk (.npz)


In [3]:
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
test_feat_path  = os.path.join(ARTIFACT_DIR, 'test_features.npz')

np.savez_compressed(train_feat_path, X_train_features=X_train_features, y_train=y_train)
np.savez_compressed(test_feat_path, X_test_features=X_test_features, y_test=y_test)

print(f"[BERHASIL] Fitur training tersimpan: {train_feat_path}")
print(f"[BERHASIL] Fitur testing tersimpan : {test_feat_path}")


[BERHASIL] Fitur training tersimpan: cifar10_artifacts/grayscale_avg\train_features.npz
[BERHASIL] Fitur testing tersimpan : cifar10_artifacts/grayscale_avg\test_features.npz
